In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

In [2]:
# Load dataset, re-format date
df = pd.read_csv("../data/processed/features_dataset.csv",parse_dates=["date"])
df = df.set_index("date").sort_index()

df.head()

,open,high,low,close,volume,fear_greed,hash_rate,n_transactions,close_lag_1,close_lag_2,...,day_of_week,day_of_month,month,quarter,is_weekend,dow_sin,dow_cos,month_sin,month_cos,target
date,,,,,,,,,,,,,,,,,,,,,
2020-02-03,9331.59,9618.79,9234.00,9292.24,50892.133451,59.0,1.137859e+08,331012.0,9331.51,9384.61,...,0,3,2,1,0,0.000000,1.000000,0.866025,0.5,9197.02
2020-02-04,9291.35,9350.00,9093.01,9197.02,53308.175266,56.0,1.014848e+08,309190.0,9292.24,9331.51,...,1,4,2,1,0,0.781831,0.623490,0.866025,0.5,9612.04
2020-02-05,9197.02,9744.45,9177.22,9612.04,64870.415615,53.0,1.222430e+08,368604.0,9197.02,9292.24,...,2,5,2,1,0,0.974928,-0.222521,0.866025,0.5,9772.00
2020-02-06,9612.03,9862.57,9526.35,9772.00,64949.706588,61.0,1.037912e+08,315370.0,9612.04,9197.02,...,3,6,2,1,0,0.433884,-0.900969,0.866025,0.5,9813.73
2020-02-07,9772.00,9885.00,9730.00,9813.73,43966.114632,56.0,1.068665e+08,334938.0,9772.00,9612.04,...,4,7,2,1,0,-0.433884,-0.900969,0.866025,0.5,9895.05


In [3]:
# Set x, y from dataset
X = df.drop("target", axis=1).values
y = df["target"].values

In [4]:
# Splitting the dataset into the Training set and Test set
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 36, shuffle= False)

In [5]:
# Model parameters
gbm = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=36,
    early_stopping_rounds=50
)

In [6]:
gbm.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=50,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=3, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=500, n_jobs=None,
             num_parallel_tree=None, random_state=36, ...)

In [14]:
pred = gbm.predict(X_test)

In [16]:
xgb_mae = np.mean(np.abs(y_test, pred))
xgb_rmse = np.sqrt(np.mean(np.abs(y_test, pred)))
xgb_mape = np.mean(np.abs((y_test - pred) / y_test)) * 100

print(f"MAE:  {xgb_mae:,.2f}")
print(f"RMSE: {xgb_rmse:,.2f}")
print(f"MAPE: {xgb_mape:.2f}")

MAE:  94,658.28
RMSE: 307.67
MAPE: 0.00
